<style>
#MathJax_Message { display: none !important; }
.odp-slide { height: 610px; box-sizing: border-box; background: white; color: #16212b; padding: 12px 38px 20px; position: relative; overflow: hidden; font-family: Arial, "Noto Sans", sans-serif; }
.odp-slide::after { content: ""; position: absolute; left: 0; right: 0; bottom: 0; height: 5px; background: linear-gradient(90deg, #04617b 0%, #009eda 60%, #8ad7ef 100%); }
.odp-slide h2 { color: #04617b; font-size: 30px; font-weight: 400; margin: 0 0 12px; padding-bottom: 12px; line-height: 1.15; position: relative; }
.odp-slide h2::after { content: ""; position: absolute; left: 0; bottom: 0; width: 74px; height: 4px; border-radius: 2px; background: #009eda; }
.odp-slide p { font-size: 21px; line-height: 1.35; }
.odp-slide li { font-size: 20px; line-height: 1.35; }
.odp-slide p { margin: 10px 0; }
.odp-slide ul { margin: 8px 0 0; padding: 0; list-style: none; }
.odp-slide li { margin: 9px 0; padding-left: 26px; position: relative; }
.odp-slide li::before { content: ""; position: absolute; left: 2px; top: 0.52em; width: 9px; height: 9px; border-radius: 50%; background: #009eda; }
.odp-slide img { object-fit: contain; }
.odp-slide .small { font-size: 18px; color: #5b666d; }
.odp-slide .metric { color: #156082; font-size: 29px; }
.odp-slide .red { color: #c9211e; }
.odp-slide .lead { color: #04617b; font-weight: 700; }
.odp-slide .two { display: flex; gap: 38px; align-items: center; }
.odp-slide table { border-collapse: collapse; width: 82%; margin: 22px 0; font-size: 19px; }
.odp-slide th { padding: 8px 13px; text-align: left; font-weight: 700; color: #04617b; border-bottom: 2px solid #04617b; white-space: nowrap; }
.odp-slide td { padding: 8px 13px; text-align: left; border-bottom: 1px solid #e2e9ec; }
.odp-slide tbody tr:nth-child(even) { background: #f2f8fb; }
.odp-slide .n { text-align: right; }
.odp-slide th.wrap { white-space: normal; line-height: 1.2; }
</style><div class="odp-slide">
<div style="width:74px; height:6px; border-radius:3px; background:#009eda; margin-top:76px;"></div>
<p style="color:#04617b; font-size:40px; line-height:1.15; margin:26px 0 0;">Alzheimer's disease classification</p>
<p class="small" style="margin:12px 0 0; font-size:21px; color:#5b666d;">Predicting the disease stage from brain MRI scans</p>
<div style="margin-top:44px;">
<p style="margin:0 0 8px;"><b>Jovan Vukićević</b></p>
<p style="margin:0;"><b>Vladeta Vujačić</b></p>
</div>
</div>

<div class="odp-slide"><h2>Problem</h2>
<ul style="margin-top:26px;">
<li>Given one brain MRI slice, predict which stage of Alzheimer's disease it shows</li>
<li>The four stages are non-demented, very mild, mild and moderate demented, in order of severity</li>
<li>They differ mainly in how far the cortex has thinned, so the classes look alike</li>
<li>The rarest stage has 64 images in the whole dataset, which is why macro F1 is the metric that matters</li>
</ul></div>

<div class="odp-slide"><h2>Dataset</h2>
<ul style="margin:0 0 4px;">
<li>6,400 grayscale MRI slices, published on Kaggle and used by the referenced paper</li>
<li>Four stages, each image a single slice rather than a whole volume</li>
<li>Heavily imbalanced, 3,200 non-demented against 64 moderate demented</li>
<li>No patient identifiers, so slices of one person cannot be kept in the same split</li>
</ul>
<img src="presentation_assets/mri_samples.svg" alt="One MRI example from each of the four classes" style="display:block; width:1000px; height:285px; margin:34px 0 0;"></div>

<div class="odp-slide"><h2>Train, validation and test data</h2>
<img src="presentation_assets/split_distribution.png" alt="Training validation and test class counts" style="display:block; width:644px; height:290px; margin:0;">
<ul style="margin:30px 0 0;">
<li>4,096 training, 1,025 validation and 1,279 test images</li>
<li>Validation is taken from the training directory alone, a fifth of it</li>
<li>The split is stratified, so every class keeps its proportion</li>
<li>The original Kaggle test directory is never touched</li>
</ul></div>

<div class="odp-slide"><h2>Problem during development - data leakage</h2>
<img src="presentation_assets/data_leakage.svg" alt="Patient leakage caused by an image-level split" style="display:block; width:980px; height:390px; margin:0;">
<ul style="margin:28px 0 0;">
<li>We combined the dataset and split individual images by hand, so slices of one patient landed in both the training and the test set</li>
</ul></div>

<div class="odp-slide"><h2>Fixing the evaluation</h2><ul>
<li>Keep the original test directory separate</li>
<li>Create validation data only from the training directory</li>
<li>Never use test images for normalization, augmentation or model selection</li>
<li class="red">Discard the earlier near-perfect scores</li>
<li>Use patient identifiers and a subject-wise split when they become available</li>
</ul></div>

<div class="odp-slide"><h2>Data preprocessing</h2>
<img src="presentation_assets/preprocessing_pipeline.svg" alt="The four preprocessing steps every image passes through" style="display:block; width:1000px; height:181px; margin:16px 0 0;">
<ul style="margin:30px 0 0;">
<li>The mean and standard deviation come from the training images alone, so nothing about validation or test reaches the model</li>
<li>Resizing to 128 × 128 keeps the whole head in view and makes the ResNet-50 runs affordable</li>
<li>Images are loaded in batches of 64</li>
</ul></div>

<div class="odp-slide"><h2>Data augmentation</h2>
<img src="presentation_assets/augmentation_examples.png" alt="Original and augmented MRI images" style="display:block; width:596px; height:289px; margin:0;">
<ul style="margin:30px 0 0;">
<li>Applied only to the two rare stages, as extra copies added to the training set</li>
<li>Each copy gets a random flip, a rotation up to 10°, a 5% shift and scale, and a brightness and contrast jitter</li>
<li>MildDemented grows from 573 to 700 and ModerateDemented from 42 to 320, giving 4,501 training images</li>
<li>Original, validation and test images are left untouched</li>
</ul></div>

<div class="odp-slide"><h2>Model - convolutional neural network</h2>
<img src="presentation_assets/cnn_architecture.svg" alt="CNN architecture" style="display:block; width:800px; height:318px; margin:4px 0 0;">
<table style="width:64%; margin:24px 0 0;">
<thead><tr>
<th align="left" style="text-align:left;">Trained on</th>
<th class="n" align="right" style="text-align:right;">Accuracy</th>
<th class="n" align="right" style="text-align:right;">Weighted F1</th>
<th class="n" align="right" style="text-align:right;">Macro F1</th>
</tr></thead>
<tbody>
<tr><td align="left" style="text-align:left;">Original images</td><td class="n" align="right" style="text-align:right;">0.686</td><td class="n" align="right" style="text-align:right;">0.672</td><td class="n" align="right" style="text-align:right;">0.493</td></tr>
<tr><td align="left" style="text-align:left;">Augmented images</td><td class="n" align="right" style="text-align:right;">0.646</td><td class="n" align="right" style="text-align:right;">0.656</td><td class="n" align="right" style="text-align:right;">0.568</td></tr>
</tbody></table></div>

<div class="odp-slide"><h2>Model - ResNet-50</h2>
<img src="presentation_assets/resnet_architecture.svg" alt="ResNet-50 architecture" style="display:block; width:800px; height:318px; margin:4px 0 0;">
<table style="width:56%; margin:24px 0 0;">
<thead><tr>
<th align="left" style="text-align:left;">Trained on</th>
<th class="n" align="right" style="text-align:right;">Accuracy</th>
<th class="n" align="right" style="text-align:right;">Weighted F1</th>
<th class="n" align="right" style="text-align:right;">Macro F1</th>
</tr></thead>
<tbody>
<tr><td align="left" style="text-align:left;">Augmented set</td><td class="n" align="right" style="text-align:right;">0.769</td><td class="n" align="right" style="text-align:right;">0.770</td><td class="n" align="right" style="text-align:right;">0.792</td></tr>
</tbody></table></div>

<div class="odp-slide"><h2>Results - CNN baselines and ResNet-50</h2>
<img src="presentation_assets/cnn_resnet_comparison.png" alt="Test comparison of the two CNNs and ResNet-50" style="display:block; width:470px; height:274px; margin:0;">
<table style="width:52%; margin:12px 0 0;">
<thead><tr><th align="left" style="text-align:left;">Model</th><th class="n" align="right" style="text-align:right;">Accuracy</th><th class="n" align="right" style="text-align:right;">Macro F1</th></tr></thead>
<tbody>
<tr><td align="left" style="text-align:left;">Non-Augmented CNN</td><td class="n" align="right" style="text-align:right;">0.686</td><td class="n" align="right" style="text-align:right;">0.493</td></tr>
<tr><td align="left" style="text-align:left;">Augmented CNN</td><td class="n" align="right" style="text-align:right;">0.646</td><td class="n" align="right" style="text-align:right;">0.568</td></tr>
<tr><td align="left" style="text-align:left;">ResNet-50</td><td class="n" align="right" style="text-align:right;">0.769</td><td class="n" align="right" style="text-align:right;">0.792</td></tr>
</tbody></table>
<ul style="margin:16px 0 0;">
<li>Augmentation raises macro F1 from 0.49 to 0.57 at the cost of accuracy, and ImageNet initialisation lifts it to 0.79</li>
</ul></div>

<div class="odp-slide"><h2>Contrastive learning</h2>
<img src="presentation_assets/contrastive_architecture.svg" alt="Contrastive learning architecture" style="display:block; width:660px; height:263px; margin:0;">
<ul style="margin:26px 0 0;">
<li>The encoder first learns which scans belong together, and a classifier is attached afterwards</li>
<li>Supervised contrastive learning counts every scan of the same stage as a match, so it needs the labels</li>
<li>SimCLR counts only the second view of the same scan, so the encoder never sees a label</li>
<li>The two views need a stronger transform than augmentation, since the encoder learns only from their differences</li>
<li>A flip, a rotation up to 15°, a 6% shift and a 0.25 brightness and contrast jitter, with no scaling</li>
</ul></div>

<div class="odp-slide"><h2>From encoder to classifier</h2>
<img src="presentation_assets/classifier_head.svg" alt="The pretrained encoder with a classification head on top" style="display:block; width:1000px; height:398px; margin:14px 0 0;"></div>

<div class="odp-slide"><h2>The five contrastive runs side by side</h2>
<table style="width:100%; margin:18px 0 0;">
<thead><tr>
<th align="left" style="text-align:left;">Model</th>
<th class="n wrap" align="right" style="text-align:right;">Labels in<br>pretraining</th>
<th class="n wrap" align="right" style="text-align:right;">Same class<br>is a match</th>
<th class="n wrap" align="right" style="text-align:right;">Encoder in<br>stage two</th>
<th class="n wrap" align="right" style="text-align:right;">Test<br>macro F1</th>
</tr></thead>
<tbody>
<tr><td align="left" style="text-align:left;">SupCon Frozen</td><td class="n" align="right" style="text-align:right;">Yes</td><td class="n" align="right" style="text-align:right;">Yes</td><td class="n" align="right" style="text-align:right;">Frozen</td><td class="n" align="right" style="text-align:right;">0.798</td></tr>
<tr><td align="left" style="text-align:left;">SupCon Fine-Tuned</td><td class="n" align="right" style="text-align:right;">Yes</td><td class="n" align="right" style="text-align:right;">Yes</td><td class="n" align="right" style="text-align:right;">Fine-tuned</td><td class="n" align="right" style="text-align:right;">0.834</td></tr>
<tr><td align="left" style="text-align:left;">SimCLR Frozen</td><td class="n" align="right" style="text-align:right;">No</td><td class="n" align="right" style="text-align:right;">No</td><td class="n" align="right" style="text-align:right;">Frozen</td><td class="n" align="right" style="text-align:right;">0.547</td></tr>
<tr><td align="left" style="text-align:left;">SimCLR Fine-Tuned</td><td class="n" align="right" style="text-align:right;">No</td><td class="n" align="right" style="text-align:right;">No</td><td class="n" align="right" style="text-align:right;">Fine-tuned</td><td class="n" align="right" style="text-align:right;">0.780</td></tr>
<tr><td align="left" style="text-align:left;">SimCLR + 10% Labels</td><td class="n" align="right" style="text-align:right;">No</td><td class="n" align="right" style="text-align:right;">No</td><td class="n" align="right" style="text-align:right;">Fine-tuned</td><td class="n" align="right" style="text-align:right;">0.413</td></tr>
</tbody></table>
<ul style="margin:20px 0 0;">
<li>The classifier stage always uses labels, and every run but the last one uses all 4,096 of them</li>
<li>Pretraining with labels is worth 0.25 while the encoder stays frozen, 0.798 against 0.547, but only 0.05 once it is fine-tuned, 0.834 against 0.780</li>
</ul></div>

<div class="odp-slide"><h2>Results - all models in numbers</h2>
<table style="width:100%; margin:16px 0 0;">
<thead><tr>
<th align="left" style="text-align:left;">Model</th>
<th class="n" align="right" style="text-align:right;">Test loss</th>
<th class="n" align="right" style="text-align:right;">Accuracy</th>
<th class="n" align="right" style="text-align:right;">W. precision</th>
<th class="n" align="right" style="text-align:right;">W. F1</th>
<th class="n" align="right" style="text-align:right;">Macro F1</th>
</tr></thead>
<tbody>
<tr><td align="left" style="text-align:left;">Non-Augmented CNN</td><td class="n" align="right" style="text-align:right;">1.4658</td><td class="n" align="right" style="text-align:right;">0.6857</td><td class="n" align="right" style="text-align:right;">0.7020</td><td class="n" align="right" style="text-align:right;">0.6718</td><td class="n" align="right" style="text-align:right;">0.4929</td></tr>
<tr><td align="left" style="text-align:left;">Augmented CNN</td><td class="n" align="right" style="text-align:right;">1.3802</td><td class="n" align="right" style="text-align:right;">0.6458</td><td class="n" align="right" style="text-align:right;">0.6782</td><td class="n" align="right" style="text-align:right;">0.6556</td><td class="n" align="right" style="text-align:right;">0.5678</td></tr>
<tr><td align="left" style="text-align:left;">Selected ResNet-50</td><td class="n" align="right" style="text-align:right;">1.1697</td><td class="n" align="right" style="text-align:right;">0.7686</td><td class="n" align="right" style="text-align:right;">0.7743</td><td class="n" align="right" style="text-align:right;">0.7698</td><td class="n" align="right" style="text-align:right;">0.7916</td></tr>
<tr><td align="left" style="text-align:left;">SupCon Frozen</td><td class="n" align="right" style="text-align:right;">1.0190</td><td class="n" align="right" style="text-align:right;">0.8178</td><td class="n" align="right" style="text-align:right;">0.8321</td><td class="n" align="right" style="text-align:right;">0.8170</td><td class="n" align="right" style="text-align:right;">0.7978</td></tr>
<tr><td align="left" style="text-align:left;">SupCon Fine-Tuned</td><td class="n" align="right" style="text-align:right;">0.9748</td><td class="n" align="right" style="text-align:right;">0.8038</td><td class="n" align="right" style="text-align:right;">0.8250</td><td class="n" align="right" style="text-align:right;">0.8054</td><td class="n" align="right" style="text-align:right;">0.8342</td></tr>
<tr><td align="left" style="text-align:left;">SimCLR Frozen</td><td class="n" align="right" style="text-align:right;">1.6207</td><td class="n" align="right" style="text-align:right;">0.6677</td><td class="n" align="right" style="text-align:right;">0.6939</td><td class="n" align="right" style="text-align:right;">0.6727</td><td class="n" align="right" style="text-align:right;">0.5466</td></tr>
<tr><td align="left" style="text-align:left;">SimCLR Fine-Tuned</td><td class="n" align="right" style="text-align:right;">0.8269</td><td class="n" align="right" style="text-align:right;">0.7780</td><td class="n" align="right" style="text-align:right;">0.8014</td><td class="n" align="right" style="text-align:right;">0.7841</td><td class="n" align="right" style="text-align:right;">0.7796</td></tr>
<tr><td align="left" style="text-align:left;">SimCLR + 10% Labels</td><td class="n" align="right" style="text-align:right;">1.8401</td><td class="n" align="right" style="text-align:right;">0.6224</td><td class="n" align="right" style="text-align:right;">0.6395</td><td class="n" align="right" style="text-align:right;">0.6136</td><td class="n" align="right" style="text-align:right;">0.4128</td></tr>
</tbody></table></div>

<div class="odp-slide"><h2>Validation against test</h2>
<table style="width:62%; margin:16px 0 0;">
<thead><tr><th align="left" style="text-align:left;">Model</th><th class="n" align="right" style="text-align:right;">Validation macro F1</th><th class="n" align="right" style="text-align:right;">Test macro F1</th><th class="n" align="right" style="text-align:right;">Drop</th></tr></thead>
<tbody>
<tr><td align="left" style="text-align:left;">Selected ResNet-50</td><td class="n" align="right" style="text-align:right;">0.989</td><td class="n" align="right" style="text-align:right;">0.792</td><td class="n" align="right" style="text-align:right;">0.197</td></tr>
<tr><td align="left" style="text-align:left;">SupCon Frozen</td><td class="n" align="right" style="text-align:right;">0.995</td><td class="n" align="right" style="text-align:right;">0.798</td><td class="n" align="right" style="text-align:right;">0.197</td></tr>
<tr><td align="left" style="text-align:left;">SupCon Fine-Tuned</td><td class="n" align="right" style="text-align:right;">0.997</td><td class="n" align="right" style="text-align:right;">0.834</td><td class="n" align="right" style="text-align:right;">0.163</td></tr>
<tr><td align="left" style="text-align:left;">SimCLR Frozen</td><td class="n" align="right" style="text-align:right;">0.915</td><td class="n" align="right" style="text-align:right;">0.547</td><td class="n" align="right" style="text-align:right;">0.368</td></tr>
<tr><td align="left" style="text-align:left;">SimCLR Fine-Tuned</td><td class="n" align="right" style="text-align:right;">0.996</td><td class="n" align="right" style="text-align:right;">0.780</td><td class="n" align="right" style="text-align:right;">0.216</td></tr>
<tr><td align="left" style="text-align:left;">SimCLR + 10% Labels</td><td class="n" align="right" style="text-align:right;">0.489</td><td class="n" align="right" style="text-align:right;">0.413</td><td class="n" align="right" style="text-align:right;">0.076</td></tr>
</tbody></table>
<ul style="margin:30px 0 0;">
<li>The slices carry no patient identifiers, so near-duplicate slices of one scan can sit in both training and validation</li>
<li>The test directory came separately from Kaggle and was never touched, so it has no such overlap</li>
<li>The paper reports the same effect, about 15% higher accuracy on validation than on test</li>
<li>The smallest drop belongs to the 10% run, which had the fewest training images that could reappear in validation</li>
</ul></div>

<div class="odp-slide"><h2>Comparison with the original paper</h2>
<table style="width:64%; margin:20px 0;">
<thead><tr>
<th align="left" style="text-align:left;">Model</th>
<th class="n" align="right" style="text-align:right;">Paper F1</th>
<th class="n" align="right" style="text-align:right;">Our macro F1</th>
</tr></thead>
<tbody>
<tr><td align="left" style="text-align:left;">ResNet-50</td><td class="n" align="right" style="text-align:right;">0.73</td><td class="n" align="right" style="text-align:right;">0.79</td></tr>
<tr><td align="left" style="text-align:left;">Supervised contrastive</td><td class="n" align="right" style="text-align:right;">0.92</td><td class="n" align="right" style="text-align:right;">0.83</td></tr>
<tr><td align="left" style="text-align:left;">SimCLR + 10% labels</td><td class="n" align="right" style="text-align:right;">0.40</td><td class="n" align="right" style="text-align:right;">0.41</td></tr>
</tbody></table>
<ul>
<li>The paper's supervised model never predicted the Moderate class, while ours reached 0.96 F1 there</li>
<li>That also shows its 0.92 cannot be a macro score, because one class at zero caps macro F1 at 0.75, so the two columns are only roughly comparable</li>
</ul>
</div>

<div class="odp-slide"><h2>Results - best model, SupCon Fine-Tuned</h2>
<div style="display:flex; gap:26px; margin:4px 0 0;">
<div style="flex:1; background:#f2f8fb; border-left:5px solid #009eda; padding:10px 18px;"><div style="font-size:33px; color:#04617b; font-weight:700; line-height:1.1;">0.834</div><div style="font-size:17px; color:#5b666d;">Macro F1</div></div>
<div style="flex:1; background:#f2f8fb; border-left:5px solid #009eda; padding:10px 18px;"><div style="font-size:33px; color:#04617b; font-weight:700; line-height:1.1;">0.804</div><div style="font-size:17px; color:#5b666d;">Accuracy</div></div>
<div style="flex:1; background:#f2f8fb; border-left:5px solid #009eda; padding:10px 18px;"><div style="font-size:33px; color:#04617b; font-weight:700; line-height:1.1;">0.960</div><div style="font-size:17px; color:#5b666d;">Moderate class F1 score</div></div>
</div>
<img src="presentation_assets/supcon_finetuned_confusion.png" alt="Confusion matrix of the fine-tuned SupCon model" style="display:block; width:502px; height:405px; margin:22px 0 0;"></div>

<div class="odp-slide"><h2>Conclusion</h2>
<ul style="margin-top:16px;">
<li>Accuracy hides what matters here, because a model can look strong while almost never recognising the two rare stages, which is why macro F1 is used throughout</li>
<li>Transferring ImageNet weights was the single largest gain of the whole project</li>
<li>Supervised contrastive pretraining improved on that again and produced the best model, the rarest class included</li>
<li>An encoder trained without any labels caught up almost completely once every label went to the classifier, so the weak result with a tenth of the labels reflects the label count rather than the pretraining</li>
<li>Every model scores far better on validation than on test, which points at slices of one patient sitting in both splits</li>
<li>A trustworthy result would need patient identifiers, a subject-wise split, repeated seeds and more examples of the rare stages</li>
</ul></div>

<div class="odp-slide"><h2>References</h2><ul>
<li><a href="https://cs230.stanford.edu/projects_fall_2022/reports/108.pdf">Fang Shu and Longling Tian, <i>Deep Learning Methods for Alzheimer's Disease Prediction</i></a></li>
<li><a href="https://www.kaggle.com/datasets/kumarln/alzheimers-disease-dataset">Kaggle - Alzheimer's Disease Dataset</a></li>
</ul></div>